<a href="https://colab.research.google.com/github/rsc-repo/IT-customer-support-Ticket-Classification-and-Routing/blob/main/notebooks/Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# -*- coding: utf-8 -*-
"""
Smart Customer Support Ticket Classification & Routing System
=============================================================
Self-contained Google Colab script.

Pipeline:
    Gradio UI -> Preprocessing (clean + PII removal) -> DistilBERT classifier
              -> Routing logic (priority + support queue)

Run in Colab (GPU runtime recommended: Runtime > Change runtime type > T4).
"""

# =============================================================================
# STEP 0: INSTALL DEPENDENCIES (safe to re-run)
# =============================================================================
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "transformers", "datasets", "accelerate",
        "scikit-learn", "gradio",
    ]
)

# =============================================================================
# STEP 1: IMPORTS & GLOBAL CONFIGURATION
# =============================================================================
import random
import re
from typing import Dict, List, Tuple

import gradio as gr
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 2
LEARNING_RATE = 2e-5
NUM_TICKETS = 2400  # >= 2,000 required

CATEGORIES = [
    "Hardware Issue",
    "Network & VPN",
    "Software Bug",
    "Account & Access",
    "Billing & Payment",
]

# Category -> designated support group (routing rules)
QUEUE_MAP = {
    "Hardware Issue": "Tier-1 Desktop Support",
    "Network & VPN": "Network Operations Center (NOC)",
    "Software Bug": "Application Support (Tier-2)",
    "Account & Access": "Identity & Access Management",
    "Billing & Payment": "Finance & Billing Support",
}

# Words that force an "Urgent" priority (whole-word, case-insensitive)
URGENT_PATTERN = re.compile(
    r"\b(crash\w*|down|error\w*|fail(?:ed|s|ing|ure|ures)?|bsod|breach\w*)\b",
    flags=re.IGNORECASE,
)

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


# =============================================================================
# STEP 2: TEXT PREPROCESSING (whitespace cleanup + regex PII removal)
# =============================================================================
# Order matters: match cards before phone numbers so digits aren't half-eaten.
_EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+")
_CARD_RE = re.compile(r"\b(?:\d[ -]?){13,16}\b")
_PHONE_RE = re.compile(r"(?<!\d)(?:\+?\d{1,3}[\s.-]?)?(?:\(?\d{3}\)?[\s.-]?)\d{3}[\s.-]?\d{4}(?!\d)")


def clean_text(text) -> str:
    """Strip whitespace, collapse spacing and mask PII (cards, phones, emails)."""
    if text is None:
        return ""
    text = str(text).strip()
    text = _EMAIL_RE.sub("[EMAIL]", text)
    text = _CARD_RE.sub("[CARD]", text)
    text = _PHONE_RE.sub("[PHONE]", text)
    text = re.sub(r"\s+", " ", text)  # collapse newlines / repeated spaces
    return text.strip()


# =============================================================================
# STEP 3: SYNTHETIC DATASET GENERATOR
# =============================================================================
# Each category has: issue templates (with {slots}), plus shared openers/closers.
OPENERS = [
    "", "", "Hi team,", "Hello,", "Urgent help needed:", "Good morning,",
    "Quick question:", "Please assist.", "Hi IT,", "Need support:",
]
CLOSERS = [
    "", "", "Please advise.", "Thanks in advance.", "Can someone look into this?",
    "This is affecting my work.", "Let me know what you need from me.",
    "It started this morning.", "Happened again today.", "Thanks!",
]

SLOTS = {
    "device": ["laptop", "desktop", "workstation", "MacBook", "ThinkPad", "Dell XPS"],
    "peripheral": ["keyboard", "mouse", "monitor", "docking station", "webcam", "headset", "printer"],
    "app": ["Outlook", "Excel", "Teams", "Zoom", "Chrome", "Salesforce", "Photoshop", "the CRM", "Slack"],
    "wifi": ["office WiFi", "guest WiFi", "home WiFi", "conference room WiFi"],
    "site": ["the intranet", "the shared drive", "the company portal", "our internal wiki", "the ERP system"],
    "code": ["0x80070005", "error 404", "code 5002", "0xC0000185", "error 1603", "code 7001"],
    "amount": ["$49.99", "$120.00", "$899", "$15.50", "$2,300", "$310.75"],
    "invoice": ["INV-20431", "INV-77812", "INV-10593", "INV-88231", "INV-45620"],
    "days": ["two days", "three days", "since Monday", "a week", "an hour", "since yesterday"],
}

TEMPLATES = {
    "Hardware Issue": [
        "My {device} will not turn on after the battery drained completely.",
        "The {peripheral} stopped working and the {device} does not detect it.",
        "My {device} screen is flickering and has a cracked corner.",
        "The {device} fan is making a loud grinding noise and it overheats.",
        "I spilled coffee on my {device} and the keys are sticking.",
        "The {peripheral} is not responding, I already tried another USB port.",
        "My {device} keeps shutting off randomly and feels very hot.",
        "The {device} charger is broken and the battery is not charging.",
        "Blue screen on my {device} every time I plug in the {peripheral}, BSOD again.",
        "The {peripheral} is physically damaged and needs a replacement.",
        "My {device} hard drive is clicking and the machine crashed twice today.",
        "There is no display output from the {device} to the {peripheral}.",
        "The {peripheral} at my desk has dead pixels and flickers constantly.",
    ],
    "Network & VPN": [
        "I cannot connect to the VPN, it says authentication timed out.",
        "The {wifi} keeps dropping every few minutes on my {device}.",
        "Internet is really slow and {site} will not load.",
        "VPN client shows connected but I cannot reach {site}.",
        "The network is down on the third floor, nobody can get online.",
        "DNS resolution fails when I try to open {site}.",
        "I get a limited connectivity warning on the {wifi} since I arrived.",
        "Remote desktop over VPN disconnects and the tunnel failed to reconnect.",
        "Ping to the office gateway is timing out and packet loss is high.",
        "Cannot access {site} from home, the VPN connection failed with {code}.",
        "The ethernet port at my desk has no link light and no network.",
        "Bandwidth is terrible during video calls, {app} keeps freezing on {wifi}.",
        "Firewall seems to be blocking my connection to {site}.",
    ],
    "Software Bug": [
        "{app} crashes every time I open a large file.",
        "I get {code} when I try to install the latest update of {app}.",
        "{app} freezes when I click the export button and I lose my work.",
        "The report generated in {app} shows wrong totals, this looks like a bug.",
        "After the update, {app} will not launch and shows an error message.",
        "The save button in {app} does nothing and the app becomes unresponsive.",
        "{app} throws {code} on startup and closes immediately.",
        "The search feature in {app} returns no results even for exact matches.",
        "Formatting breaks when I paste content into {app}, it started after the patch.",
        "{app} keeps showing a blank white window after login.",
        "Please fix the bug where {app} duplicates records when I sync.",
        "The installer for {app} failed at 90 percent with {code}.",
        "Dark mode in {app} makes the text unreadable, please fix.",
    ],
    "Account & Access": [
        "I forgot my password and the reset email never arrives.",
        "My account is locked after too many failed login attempts.",
        "I need access to {site} for my new project role.",
        "Please reset my multi-factor authentication, I got a new phone.",
        "I cannot log into {app}, it says my credentials are invalid.",
        "A new hire needs an account and mailbox created before Monday.",
        "My permissions on the shared folder were removed and I am blocked.",
        "Please disable the account of a departing employee effective today.",
        "The single sign-on page keeps looping back to the login screen.",
        "I am not able to approve requests because my admin role is missing.",
        "Need my username changed after my name update.",
        "Access denied when opening {site}, I should have read permissions.",
        "Two-factor codes are not being accepted for my account.",
    ],
    "Billing & Payment": [
        "I was charged {amount} twice for the same subscription.",
        "Invoice {invoice} shows the wrong amount, it should be {amount}.",
        "My credit card was declined when I tried to renew the license.",
        "Please send me a refund for the payment of {amount} made {days} ago.",
        "I did not receive the receipt for invoice {invoice}.",
        "Can you update the billing address on my account?",
        "I was billed after I cancelled my plan, please reverse the charge of {amount}.",
        "The payment page shows an error when I enter my card details.",
        "I need a copy of invoice {invoice} for our accounting department.",
        "Why was my monthly fee increased to {amount}? Please explain the charges.",
        "The tax on invoice {invoice} looks incorrect.",
        "My payment was taken but the subscription is still marked as unpaid.",
        "Please switch my billing from monthly to annual and apply the discount.",
    ],
}


def _fill_slots(template: str) -> str:
    """Replace every {slot} in a template with a random value."""
    def repl(match: re.Match) -> str:
        return random.choice(SLOTS[match.group(1)])
    return re.sub(r"\{(\w+)\}", repl, template)


def _light_noise(text: str) -> str:
    """Apply small realistic variations (casing, trailing punctuation)."""
    roll = random.random()
    if roll < 0.10:
        text = text.lower()
    elif roll < 0.15:
        text = text.upper()
    if random.random() < 0.10:
        text = text.rstrip(".") + "!!"
    return text


def generate_dataset(n_samples: int = NUM_TICKETS) -> pd.DataFrame:
    """Generate a balanced synthetic ticket dataset as a DataFrame."""
    rows: List[Tuple[str, str]] = []
    per_class = n_samples // len(CATEGORIES)
    for category in CATEGORIES:
        for _ in range(per_class):
            body = _fill_slots(random.choice(TEMPLATES[category]))
            parts = [random.choice(OPENERS), body, random.choice(CLOSERS)]
            text = _light_noise(" ".join(p for p in parts if p))
            rows.append((text, category))
    df = pd.DataFrame(rows, columns=["text", "category"])
    return df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)


df = generate_dataset()
df["text"] = df["text"].apply(clean_text)
print(f"Generated {len(df)} tickets. Sample:")
print(df.sample(5, random_state=SEED).to_string(index=False))

# =============================================================================
# STEP 4: LABEL ENCODING & 80/20 TRAIN/VALIDATION SPLIT
# =============================================================================
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["category"])
CLASS_NAMES = list(label_encoder.classes_)  # alphabetical order
id2label = {i: name for i, name in enumerate(CLASS_NAMES)}
label2id = {name: i for i, name in id2label.items()}

train_df, val_df = train_test_split(
    df[["text", "label"]],
    test_size=0.20,
    random_state=SEED,
    stratify=df["label"],
)
print(f"Train: {len(train_df)} | Validation: {len(val_df)}")

# =============================================================================
# STEP 5: TOKENIZATION (DistilBertTokenizerFast, max length 128)
# =============================================================================
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)


def tokenize_batch(batch):
    """Tokenize a batch with max-length padding and truncation."""
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
    )


train_ds = Dataset.from_pandas(train_df.reset_index(drop=True)).map(tokenize_batch, batched=True)
val_ds = Dataset.from_pandas(val_df.reset_index(drop=True)).map(tokenize_batch, batched=True)

# Keep only what the model needs
cols = ["input_ids", "attention_mask", "label"]
train_ds.set_format("torch", columns=cols)
val_ds.set_format("torch", columns=cols)

# =============================================================================
# STEP 6: MODEL FINE-TUNING WITH HUGGING FACE TRAINER
# =============================================================================
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(CLASS_NAMES),
    id2label=id2label,
    label2id=label2id,
)


def compute_metrics(eval_pred):
    """Accuracy and weighted F1 for the validation set."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }


training_args = TrainingArguments(
    output_dir="./ticket_classifier",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy="epoch",          # current syntax (NOT evaluation_strategy)
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=torch.cuda.is_available(),  # mixed precision only on CUDA
    logging_steps=25,
    report_to="none",                # disable wandb etc.
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()
print("Final evaluation:", trainer.evaluate())

# Switch to inference mode
model.to(DEVICE)
model.eval()

# =============================================================================
# STEP 7: ROUTING ENGINE
# =============================================================================
_ERROR_CODE_RE = re.compile(r"\b(0x[0-9a-f]{6,8}|error \d{3,4}|code \d{4})\b", re.IGNORECASE)
_INVOICE_RE = re.compile(r"\bINV-\d{4,6}\b", re.IGNORECASE)
_APP_NAMES = [s for s in SLOTS["app"] if not s.startswith("the ")]


def extract_entities(text: str) -> Dict[str, List[str]]:
    """Lightweight rule-based key-entity extraction."""
    apps = [a for a in _APP_NAMES if re.search(rf"\b{re.escape(a)}\b", text, re.IGNORECASE)]
    return {
        "error_codes": sorted(set(m.lower() for m in _ERROR_CODE_RE.findall(text))),
        "invoice_ids": sorted(set(m.upper() for m in _INVOICE_RE.findall(text))),
        "applications": apps,
    }


def assign_priority(text: str, top_category: str, confidence: float) -> Tuple[str, str]:
    """Return (priority, reason) using keyword + category + confidence rules."""
    kw = URGENT_PATTERN.search(text)
    if kw:
        return "Urgent", f"Critical keyword detected: '{kw.group(0).lower()}'"
    if top_category == "Network & VPN":
        return "Urgent", "Category is Network & VPN"
    if confidence > 0.80:
        return "High", f"High model confidence ({confidence:.2f} > 0.80)"
    return "Normal", "Standard baseline priority"


@torch.no_grad()
def route_ticket(ticket_text: str):
    """
    Classify and route a support ticket.

    Returns
    -------
    top3 : dict   {category: probability} for the 3 most likely categories.
    meta : dict   Routing metadata (priority, queue, confidence, entities...).
    """
    cleaned = clean_text(ticket_text)

    # Fallback: empty input must never raise an exception
    if not cleaned:
        return {}, {"status": "error", "message": "Empty ticket. Please enter a description."}

    try:
        # a) tokenize
        enc = tokenizer(
            cleaned,
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        ).to(DEVICE)

        # b) inference -> Softmax probabilities
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1).squeeze(0).cpu().numpy()

        # c) top-3 predictions
        top_idx = np.argsort(probs)[::-1][:3]
        top3 = {id2label[int(i)]: round(float(probs[i]), 4) for i in top_idx}

        best_category = id2label[int(top_idx[0])]
        confidence = float(probs[top_idx[0]])

        # d) priority + queue
        priority, reason = assign_priority(cleaned, best_category, confidence)
        meta = {
            "status": "ok",
            "predicted_category": best_category,
            "confidence": round(confidence, 4),
            "priority": priority,
            "priority_reason": reason,
            "target_queue": QUEUE_MAP[best_category],
            "entities": extract_entities(cleaned),
            "cleaned_text": cleaned,
        }
        return top3, meta

    except Exception as exc:  # defensive: UI should never crash
        return {}, {"status": "error", "message": f"Inference failed: {exc}"}


# Quick smoke test
print(route_ticket("My VPN keeps failing with error 1603 and I cannot reach the shared drive."))

# =============================================================================
# STEP 8: GRADIO DASHBOARD
# =============================================================================
SAMPLE_TICKETS = {
    "Laptop won't boot": "My ThinkPad will not turn on and the charger light is off. It crashed yesterday.",
    "VPN issue": "I cannot connect to the VPN from home, the connection timed out.",
    "App bug": "Excel freezes whenever I click export and shows error 1603.",
    "Locked account": "My account is locked after too many login attempts, please reset my password.",
    "Double charge": "I was charged $49.99 twice on invoice INV-20431, please refund one payment.",
}

with gr.Blocks(title="Smart Ticket Router") as demo:
    gr.Markdown(
        "# Smart Customer Support Ticket Classification & Routing\n"
        "Paste a support ticket or click a sample. The model predicts the category, "
        "priority and target support queue."
    )

    with gr.Row():
        with gr.Column(scale=1):
            ticket_box = gr.Textbox(
                label="Ticket content",
                placeholder="Describe the issue...",
                lines=6,
            )
            submit_btn = gr.Button("Classify & Route", variant="primary")
            gr.Markdown("**Sample tickets**")
            sample_buttons = {name: gr.Button(name, size="sm") for name in SAMPLE_TICKETS}

        with gr.Column(scale=1):
            label_out = gr.Label(num_top_classes=3, label="Top-3 predictions")
            json_out = gr.JSON(label="Routing metadata")

    # Main action
    submit_btn.click(route_ticket, inputs=ticket_box, outputs=[label_out, json_out])
    ticket_box.submit(route_ticket, inputs=ticket_box, outputs=[label_out, json_out])

    # Sample buttons: fill textbox, then run routing
    for name, button in sample_buttons.items():
        button.click(
            lambda text=SAMPLE_TICKETS[name]: text, outputs=ticket_box
        ).then(route_ticket, inputs=ticket_box, outputs=[label_out, json_out])

demo.launch(inline=True, share=True, debug=False)

Using device: cuda
Generated 2400 tickets. Sample:
                                                                                                     text          category
Good morning, Bandwidth is terrible during video calls, Salesforce keeps freezing on office WiFi. Thanks!     Network & VPN
                                  I get a limited connectivity warning on the guest WiFi since I arrived.     Network & VPN
      Need support: Invoice INV-10593 shows the wrong amount, it should be $120.00. Happened again today. Billing & Payment
PLEASE ASSIST. MY PAYMENT WAS TAKEN BUT THE SUBSCRIPTION IS STILL MARKED AS UNPAID. HAPPENED AGAIN TODAY. Billing & Payment
  Hello, The mouse stopped working and the desktop does not detect it. Let me know what you need from me.    Hardware Issue
Train: 1920 | Validation: 480


Map:   0%|          | 0/1920 [00:00<?, ? examples/s]

Map:   0%|          | 0/480 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,0.184942,0.053205,1.000000,1.000000
2,0.030002,0.022558,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1 Weighted
0.030002,0.053205,2,1.000000,1.000000


Final evaluation: {'eval_loss': 0.05320478975772858, 'eval_accuracy': 1.0, 'eval_f1_weighted': 1.0}
({'Network & VPN': 0.9474, 'Account & Access': 0.0152, 'Hardware Issue': 0.0139}, {'status': 'ok', 'predicted_category': 'Network & VPN', 'confidence': 0.9474, 'priority': 'Urgent', 'priority_reason': "Critical keyword detected: 'failing'", 'target_queue': 'Network Operations Center (NOC)', 'entities': {'error_codes': ['error 1603'], 'invoice_ids': [], 'applications': []}, 'cleaned_text': 'My VPN keeps failing with error 1603 and I cannot reach the shared drive.'})
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c036d383704e8cd5c7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
